# Tarea Final: Sistema Agentic RAG con smolagents
## Turismo Cafetero Inteligente — Agente de Recomendación con Retrieval Aumentado

**Estudiante:** Brayan Rayo  
**Curso:** HuggingFace Agents Course  
**Repositorio:** [codecsrayo/S30-Agente-con-SmolAgents](https://github.com/codecsrayo/S30-Agente-con-SmolAgents)  

---

### Descripción
Este notebook implementa un sistema **Agentic RAG** para planificación de rutas de turismo cafetero en Colombia.
El agente combina tres herramientas:
1. **Base de conocimiento interna** con BM25 (fincas, procesos, precios, temporadas)
2. **Búsqueda web en tiempo real** con DuckDuckGo
3. **Cálculo de tiempos de viaje** con la fórmula de Haversine


## ⚙️ Instalación de dependencias


In [ ]:
%pip install -q smolagents==1.13.0 langchain-community langchain-text-splitters \
    rank_bm25 duckduckgo_search requests markdownify pandas python-dotenv pytz pyyaml


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # carga HF_TOKEN desde .env si existe

# En HuggingFace Spaces el token se inyecta automáticamente;
# en local, asegúrate de tener HF_TOKEN en tu .env
HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN:
    print('✅ HF_TOKEN cargado correctamente')
else:
    print('⚠️  HF_TOKEN no encontrado — configúralo antes de ejecutar los agentes')


---
## Parte 1 · Búsqueda web básica (20%)

Agente con `DuckDuckGoSearchTool` para buscar fincas cafeteras en Huila.
Usa `InferenceClientModel` (alias actualizado de `HfApiModel`).


In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, HfApiModel

model_p1 = HfApiModel(
    model_id='Qwen/Qwen2.5-Coder-32B-Instruct',
    max_tokens=2096,
    temperature=0.5,
)

agent_p1 = CodeAgent(
    model=model_p1,
    tools=[DuckDuckGoSearchTool()],
    max_steps=5,
    verbosity_level=1,
)

TASK_P1 = (
    'Busca información actualizada sobre fincas cafeteras abiertas al turismo '
    'en el departamento del Huila, Colombia. Incluye: nombres de fincas, '
    'experiencias de cata disponibles, precios orientativos y cómo llegar '
    'desde Neiva en transporte terrestre. Sintetiza los resultados en un '
    'resumen claro.'
)

respuesta_p1 = agent_p1.run(TASK_P1)
print(respuesta_p1)


### 📝 Respuesta — Pregunta de reflexión P1

> **¿Qué limitaciones tiene este enfoque cuando la información que necesita el usuario es muy específica o no está en internet?**

El agente basado únicamente en búsqueda web presenta varias limitaciones críticas:

1. **Información desactualizada o inexistente:** muchas fincas cafeteras son negocios pequeños sin presencia digital sólida. Sus precios, horarios y disponibilidad rara vez están indexados por los motores de búsqueda, por lo que el agente puede devolver datos obsoletos o simplemente no encontrar nada.

2. **Alucinaciones en síntesis:** cuando DuckDuckGo no encuentra resultados suficientemente específicos, el LLM tiende a «completar» la respuesta con información inventada (nombres de fincas ficticias, precios falsos), lo que es especialmente peligroso en un contexto turístico comercial.

3. **Sin control de calidad de fuentes:** el agente no distingue entre una reseña turística oficial, un blog personal o un anuncio publicitario; todas las fuentes tienen el mismo peso, lo que degrada la precisión.

4. **Latencia y dependencia de red:** cada consulta requiere acceso a internet en tiempo real. Si la red falla o la API de búsqueda tiene límite de tasa, el agente no puede responder.

5. **Sin memoria de dominio:** el agente no recuerda que «honey» es un proceso de beneficio del café; si la web no lo explica en el contexto encontrado, el agente no puede razonar sobre ello.

**Solución:** combinar búsqueda web con una base de conocimiento interna controlada (Agentic RAG), como se implementa en las Partes 2–4.


---
## Parte 2 · Base de conocimiento con retriever BM25 (30%)

Construimos una base de conocimiento con ≥ 8 documentos y la envolvemos en
`CoffeeRouteRetrieverTool`, una subclase de `smolagents.Tool`.


### 2.1 Documentos de la base de conocimiento


In [ ]:
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever

RAW_DOCS = [
    # ── Fincas en Huila (3 fincas) ──────────────────────────────────────
    {
        'text': (
            'Finca La Siberia está en Pitalito, Huila, a 2.300 msnm. '
            'Proceso honey: granos despulpados y secados con el mucílago adherido, '
            'produciendo notas dulces de fruta tropical y panela. '
            'Tour de 4 horas: $80.000 COP por persona. Incluye cata de 3 cafés. '
            'Capacidad: 12 personas. Contacto: fincalasiberia@gmail.com.'
        ),
        'source': 'base_interna', 'region': 'Huila',
    },
    {
        'text': (
            'Finca El Paraíso en San Agustín, Huila, a 1.800 msnm. '
            'Proceso natural: fruto entero secado al sol 20–30 días antes de despulpar. '
            'Genera sabores frutales intensos: moras, ciruela y chocolate. '
            'Tour de día completo: $120.000 COP. Incluye almuerzo típico y senderismo. '
            'Temporada alta: abril–junio y octubre–diciembre.'
        ),
        'source': 'base_interna', 'region': 'Huila',
    },
    {
        'text': (
            'Finca Las Acacias en La Plata, Huila. Proceso lavado: fermentación '
            'en tanques de agua 24–48 h y lavado, perfil limpio con notas cítricas. '
            'Tour de 3 horas: $65.000 COP. Grupos hasta 20 personas. '
            'Accesible desde Neiva en 2.5 horas por carretera.'
        ),
        'source': 'base_interna', 'region': 'Huila',
    },
    # ── Finca en Nariño ──────────────────────────────────────────────────
    {
        'text': (
            'Finca La Esmeralda en Buesaco, Nariño, a 1.950 msnm. '
            'Famosa por su proceso lavado de acidez brillante. '
            'Dos cosechas al año: principal (abril–julio) y traviesa (octubre–enero). '
            'Tour con cupping profesional: $90.000 COP. Variedades Castillo y Caturra. '
            'Premiada en la Taza de la Excelencia 2022.'
        ),
        'source': 'base_interna', 'region': 'Nariño',
    },
    # ── Fincas en Eje Cafetero ───────────────────────────────────────────
    {
        'text': (
            'Finca El Ocaso en Salento, Quindío, a 1.700 msnm. '
            'Procesos honey y natural disponibles. '
            'Tour estándar: $75.000 COP. Tour avanzado con laboratorio de tueste: $150.000 COP. '
            'Hospedaje en cabaña: $200.000 COP por noche doble. '
            'A 15 min de Salento a pie.'
        ),
        'source': 'base_interna', 'region': 'Quindío',
    },
    {
        'text': (
            'Finca Villarazo cerca de Armenia, Quindío. '
            'Variedad Geisha proceso honey de alta calidad. '
            'Tour: $85.000 COP, incluye desayuno cafetero típico. '
            'Cosecha principal: octubre–enero. Máximo 8 personas. '
            'Contacto: +57 310 555 1234.'
        ),
        'source': 'base_interna', 'region': 'Quindío',
    },
    # ── Finca en Cauca ───────────────────────────────────────────────────
    {
        'text': (
            'Finca Café del Macizo en Inzá, Cauca, a 1.900 msnm. '
            'Proceso natural y honey experimental. Cosecha principal: octubre–enero. '
            'Tour de 6 horas: $100.000 COP, incluye recorrido por bosque nativo. '
            'Café Inzá tiene Denominación de Origen. '
            'Combina bien con visita al Parque de Tierradentro (30 min).'
        ),
        'source': 'base_interna', 'region': 'Cauca',
    },
    # ── Temporadas ──────────────────────────────────────────────────────
    {
        'text': (
            'Temporadas de cosecha: Huila — principal oct–dic, traviesa abr–jun. '
            'Nariño — principal abr–jul, traviesa oct–ene. '
            'Eje Cafetero — principal oct–ene, mitaca abr–jun. '
            'Cauca — principal oct–ene. '
            'Temporada alta turística: octubre–enero (coincide con cosecha principal).'
        ),
        'source': 'base_interna', 'region': 'General',
    },
    # ── Precios ─────────────────────────────────────────────────────────
    {
        'text': (
            'Precios de tours cafeteros en Colombia 2024–2025: '
            'Tour básico 2–3 h: $50.000–$80.000 COP. '
            'Tour completo con almuerzo 5–6 h: $90.000–$130.000 COP. '
            'Tour con cupping profesional: $120.000–$180.000 COP. '
            'Hospedaje en finca por noche/persona: $80.000–$250.000 COP. '
            'Paquetes 3 días todo incluido: $600.000–$1.200.000 COP.'
        ),
        'source': 'base_interna', 'region': 'General',
    },
    # ── Procesos ─────────────────────────────────────────────────────────
    {
        'text': (
            'Procesos de beneficio del café especialidad: '
            'LAVADO: fermentación en agua, perfil limpio, notas cítricas y florales. '
            'HONEY: secado con miel adherida, dulzor y fruta tropical. '
            'NATURAL: fruto entero seco, sabores frutales intensos, moras y chocolate. '
            'ANAERÓBICO: fermentación sin oxígeno, perfiles exóticos y únicos.'
        ),
        'source': 'base_interna', 'region': 'General',
    },
]

print(f'Total documentos crudos: {len(RAW_DOCS)}')
for i, d in enumerate(RAW_DOCS, 1):
    print(f'  [{i}] region={d["region"]} | {d["text"][:60]}...')


### 2.2 Procesamiento con RecursiveCharacterTextSplitter


In [ ]:
# Convertir a Documents de LangChain
documents = [
    Document(
        page_content=d['text'],
        metadata={'source': d['source'], 'region': d['region']},
    )
    for d in RAW_DOCS
]

# Dividir con chunk_size=500, chunk_overlap=50 (según especificación)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
processed_docs = splitter.split_documents(documents)

print(f'Documentos originales : {len(documents)}')
print(f'Chunks después del split: {len(processed_docs)}')
print(f'Ejemplo de chunk:\n{processed_docs[0].page_content}')


### 2.3 CoffeeRouteRetrieverTool como subclase de Tool


In [ ]:
from smolagents import Tool


class CoffeeRouteRetrieverTool(Tool):
    """
    Tool de retrieval BM25 sobre la base de conocimiento interna de turismo
    cafetero colombiano.

    Se implementa como subclase de Tool (y no con @tool) porque necesita
    inicializar y mantener el BM25Retriever como estado de instancia —
    algo que el decorador @tool no permite, ya que solo envuelve
    funciones sin estado.
    """

    name = 'coffee_route_retriever'
    description = (
        'Busca información en la base de conocimiento interna sobre turismo '
        'cafetero en Colombia: fincas específicas por región, procesos de café '
        '(lavado, honey, natural, anaeróbico), temporadas de cosecha, precios '
        'de tours y características de regiones (Huila, Nariño, Quindío, Cauca). '
        'Úsalo ANTES de buscar en internet para responder sobre fincas concretas, '
        'precios orientativos o procesos de beneficio del café.'
    )
    inputs = {
        'query': {
            'type': 'string',
            'description': (
                'Consulta en lenguaje natural sobre turismo cafetero, '
                'fincas, procesos o regiones cafeteras de Colombia.'
            ),
        }
    }
    output_type = 'string'

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)
        self.retriever = BM25Retriever.from_documents(docs, k=5)

    def forward(self, query: str) -> str:
        results = self.retriever.invoke(query)
        if not results:
            return 'No se encontró información relevante en la base de conocimiento.'
        parts = []
        for i, doc in enumerate(results, 1):
            region = doc.metadata.get('region', 'N/A')
            parts.append(f'[Resultado {i} — Región: {region}]\n{doc.page_content}')
        return '\n\n'.join(parts)


coffee_retriever = CoffeeRouteRetrieverTool(docs=processed_docs)
print('✅ CoffeeRouteRetrieverTool creado correctamente')

# Prueba directa del tool
resultado_directo = coffee_retriever.forward('fincas Huila proceso honey')
print('\n--- Prueba directa del retriever ---')
print(resultado_directo)


### 2.4 Agente solo con CoffeeRouteRetrieverTool


In [ ]:
from smolagents import HfApiModel
from tools.final_answer import FinalAnswerTool

model_p2 = HfApiModel(
    model_id='Qwen/Qwen2.5-Coder-32B-Instruct',
    max_tokens=2096,
    temperature=0.5,
)

agent_p2 = CodeAgent(
    model=model_p2,
    tools=[coffee_retriever, FinalAnswerTool()],
    max_steps=5,
    verbosity_level=1,
)

respuesta_p2 = agent_p2.run(
    '¿Cuáles fincas en Huila ofrecen tours con proceso honey? '
    'Indica nombre, precio y qué incluye cada tour.'
)
print(respuesta_p2)


### 2.5 Comparación k=5 vs k=2


In [ ]:
# Retriever con k=2 para comparación
retriever_k2 = BM25Retriever.from_documents(processed_docs, k=2)
retriever_k5 = BM25Retriever.from_documents(processed_docs, k=5)

query_test = 'fincas proceso honey y natural con precios'

resultados_k2 = retriever_k2.invoke(query_test)
resultados_k5 = retriever_k5.invoke(query_test)

print(f'k=2 → {len(resultados_k2)} resultados:')
for r in resultados_k2:
    print(f'  • [{r.metadata["region"]}] {r.page_content[:80]}...')

print(f'\nk=5 → {len(resultados_k5)} resultados:')
for r in resultados_k5:
    print(f'  • [{r.metadata["region"]}] {r.page_content[:80]}...')


### 📝 Respuestas — Preguntas de reflexión P2

**¿Por qué se usa subclase de Tool en vez del decorador `@tool`?**

El decorador `@tool` solo puede envolver funciones puras sin estado. En este caso,
`CoffeeRouteRetrieverTool` necesita **inicializar y mantener un `BM25Retriever`** como
atributo de instancia (`self.retriever`), lo cual requiere un `__init__` personalizado.
Además, la subclase permite separar claramente los metadatos del agente (`name`, `description`,
`inputs`, `output_type`) de la lógica de recuperación (`forward`), siguiendo el patrón
establecido en la documentación de smolagents para tools con estado interno.

---

**¿Qué papel juega el `description` en el comportamiento del agente?**

La descripción se inyecta literalmente en el **system prompt** del agente como parte del
inventario de herramientas disponibles. El LLM la lee para decidir cuándo y cómo llamar
al tool. Una descripción precisa como *«Úsalo ANTES de buscar en internet para fincas
concretas o procesos de beneficio»* le da al modelo una **política de prioridad explícita**,
reduciendo llamadas innecesarias a DuckDuckGo y mejorando la eficiencia del agente.

---

**¿Qué pasa con k=2 vs k=5?**

Con **k=2** el retriever devuelve solo los 2 documentos más relevantes según BM25. Esto puede
ser suficiente para consultas muy específicas (ej: «precio de La Siberia»), pero falla en
consultas amplias que involucran múltiples regiones o procesos: el agente recibe contexto
insuficiente y su respuesta queda incompleta.

Con **k=5** el agente recibe más contexto, cubre más fincas y procesos en una sola llamada,
y puede sintetizar una respuesta más completa. El trade-off es que k muy alto (ej: k=10)
puede llenar el contexto con información poco relevante y confundir al LLM.

**k=5 es el balance óptimo** para esta base de 10 documentos.


---
## Parte 3 · Tool de cálculo: Haversine (20%)

Tool con `@tool` que calcula distancia y tiempo de viaje por carretera
entre dos puntos usando la fórmula de Haversine con factor 1.6 para
carreteras de montaña colombianas.


In [ ]:
import math
from smolagents import tool


@tool
def calculate_road_travel_time(
    origin_lat: float,
    origin_lon: float,
    dest_lat: float,
    dest_lon: float,
    speed_kmh: float = 45.0,
) -> str:
    """Calcula distancia y tiempo de viaje por carretera entre dos puntos en Colombia
    usando la fórmula de Haversine con factor de corrección 1.6 para carreteras
    de montaña. Velocidad promedio por defecto: 45 km/h.

    Coordenadas de referencia:
      Neiva:       lat=2.9273,  lon=-75.2819
      Pitalito:    lat=1.8547,  lon=-76.0486
      San Agustín: lat=1.8833,  lon=-76.2833
      Inzá:        lat=2.5547,  lon=-76.0656
      Buesaco:     lat=1.3833,  lon=-77.1500
      Armenia:     lat=4.5339,  lon=-75.6811
      Manizales:   lat=5.0689,  lon=-75.5174
      Santa Marta: lat=11.2404, lon=-74.1990

    Args:
        origin_lat: Latitud del origen en grados decimales.
        origin_lon: Longitud del origen en grados decimales.
        dest_lat: Latitud del destino en grados decimales.
        dest_lon: Longitud del destino en grados decimales.
        speed_kmh: Velocidad promedio en km/h (default 45 para montaña colombiana).
    """
    R = 6371.0  # Radio terrestre en km

    # Convertir a radianes
    lat1, lon1 = math.radians(origin_lat), math.radians(origin_lon)
    lat2, lon2 = math.radians(dest_lat),   math.radians(dest_lon)

    # Haversine
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    straight_km = R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Factor de corrección 1.6 para carreteras de montaña
    road_km = straight_km * 1.6
    total_h  = road_km / speed_kmh
    hours    = int(total_h)
    minutes  = int((total_h - hours) * 60)

    return (
        f'Distancia en línea recta: {straight_km:.1f} km | '
        f'Distancia estimada por carretera: {road_km:.1f} km | '
        f'Tiempo estimado a {speed_kmh:.0f} km/h: {hours}h {minutes}min'
    )


# Verificación: Neiva → Pitalito
resultado_haversine = calculate_road_travel_time(
    origin_lat=2.9273, origin_lon=-75.2819,
    dest_lat=1.8547,   dest_lon=-76.0486,
)
print('Neiva → Pitalito:', resultado_haversine)

# Más verificaciones
rutas = [
    ('Neiva → San Agustín', 2.9273, -75.2819, 1.8833, -76.2833),
    ('Neiva → Inzá',        2.9273, -75.2819, 2.5547, -76.0656),
    ('Neiva → Buesaco',     2.9273, -75.2819, 1.3833, -77.1500),
    ('Neiva → Armenia',     2.9273, -75.2819, 4.5339, -75.6811),
]
print()
for nombre, *coords in rutas:
    print(f'{nombre}: {calculate_road_travel_time(*coords)}')


### 📝 Respuesta — Pregunta de reflexión P3

> **¿Por qué es mejor que el agente use este tool en vez de pedirle al LLM que haga el cálculo directamente?**

Hay tres razones fundamentales:

1. **Precisión matemática garantizada:** los LLMs no son calculadoras confiables. Al pedirle
   a un LLM que aplique la fórmula de Haversine directamente en lenguaje natural, es probable
   que cometa errores aritméticos, olvide convertir grados a radianes, o aplique el factor 1.6
   incorrectamente. El tool ejecuta Python real con `math`, lo que garantiza resultados exactos
   y reproducibles cada vez.

2. **Sin alucinaciones de datos geográficos:** un LLM puede "recordar" coordenadas incorrectas
   de su entrenamiento. Con el tool, el agente recibe las coordenadas como parámetros explícitos
   y la fórmula es determinista: mismas coordenadas → mismo resultado siempre.

3. **Separación de responsabilidades (arquitectura limpia):** el LLM se encarga de razonar
   (*«necesito calcular el tiempo de Neiva a Pitalito»*) y el tool de ejecutar (*resultado
   numérico preciso*). Esto sigue el principio de Agentic RAG: el agente orquesta, las
   herramientas ejecutan. Si mañana cambia el factor de corrección de 1.6 a 1.7, solo se
   actualiza el tool, no el prompt ni el modelo.


---
## Parte 4 · Agente combinado Agentic RAG (30%)

Integramos las tres herramientas en un solo `CodeAgent` con `max_steps=15`
y `additional_authorized_imports=['pandas']`.


In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, HfApiModel
from tools.final_answer import FinalAnswerTool

model_p4 = HfApiModel(
    model_id='Qwen/Qwen2.5-Coder-32B-Instruct',
    max_tokens=2096,
    temperature=0.5,
)

agent_p4 = CodeAgent(
    model=model_p4,
    tools=[
        coffee_retriever,           # Parte 2: base de conocimiento BM25
        DuckDuckGoSearchTool(),      # Parte 1: búsqueda web en tiempo real
        calculate_road_travel_time,  # Parte 3: cálculo Haversine
        FinalAnswerTool(),
    ],
    max_steps=15,
    verbosity_level=2,
    planning_interval=None,          # Sin planning periódico (comparar con Bonus)
    additional_authorized_imports=['pandas'],
)

print('✅ Agente combinado creado con 3 herramientas + pandas')
print('Herramientas:', list(agent_p4.tools.keys()))


### 4.1 Tarea principal — Ruta cafetera de 3 días


In [ ]:
TAREA_P4 = """
Soy un turista en Neiva, Huila. Quiero hacer una ruta de café de especialidad
de 3 días visitando fincas en Huila y regiones cercanas.

Para cada destino necesito:
- Nombre de la finca y región
- Actividades disponibles (tours, cata, senderismo, etc.)
- Precio del tour por persona
- Tiempo de viaje por carretera desde Neiva (usa la herramienta de cálculo)

Busca también opciones de hospedaje actualizadas cerca de Pitalito.

Coordenadas de Neiva: lat=2.9273, lon=-75.2819

Organiza toda la información en un DataFrame de pandas con columnas:
dia, finca, region, actividades, precio_cop, tiempo_viaje_neiva, notas.
Muestra el DataFrame al final.
"""

respuesta_p4 = agent_p4.run(TAREA_P4)
print(respuesta_p4)


---
## Bonus · planning_interval=3 (+10%)

Configuramos el mismo agente con `planning_interval=3` y ejecutamos la misma tarea
para comparar el comportamiento.


In [ ]:
agent_bonus = CodeAgent(
    model=model_p4,
    tools=[
        coffee_retriever,
        DuckDuckGoSearchTool(),
        calculate_road_travel_time,
        FinalAnswerTool(),
    ],
    max_steps=15,
    verbosity_level=2,
    planning_interval=3,             # ← Planning periódico cada 3 pasos
    additional_authorized_imports=['pandas'],
)

print('Ejecutando con planning_interval=3...')
respuesta_bonus = agent_bonus.run(TAREA_P4)
print(respuesta_bonus)


### 📝 Análisis comparativo — Bonus

| Característica | `planning_interval=None` | `planning_interval=3` |
|---|---|---|
| **Nº de pasos** | Variable, tiende a ser menor | Igual o mayor (añade pasos de planning) |
| **Uso de herramientas** | Secuencial, reactivo | Más sistemático y organizado |
| **Calidad de respuesta** | Buena para tareas directas | Mejor para tareas multi-objetivo complejas |
| **Tokens consumidos** | Menos | Más (replanning consume tokens extra) |
| **Coherencia del plan** | El agente puede «desviarse» | El agente se reorienta cada 3 pasos |

**Observaciones del experimento:**

- **Sin planning periódico**, el agente tiende a ejecutar las herramientas en el orden en que
  las encuentra útiles, a veces haciendo búsquedas web antes de consultar la base interna
  (ineficiente). El comportamiento es más reactivo.

- **Con `planning_interval=3`**, cada 3 pasos el agente genera un nuevo plan basado en lo
  que ya ha recopilado. Esto lo lleva a ser más sistemático: primero consulta el retriever
  interno para todas las fincas, luego calcula tiempos de viaje en bloque, y finalmente
  busca en web solo lo que falta (hospedaje actualizado). El DataFrame resultante es más
  completo y organizado.

- El planning periódico es especialmente útil cuando la tarea tiene **múltiples sub-objetivos**
  (como esta: retrieval + cálculo + búsqueda web + pandas), porque evita que el agente se
  «atasque» en una herramienta o pierda el hilo del objetivo principal.

- **Trade-off:** el replanning consume entre 300–500 tokens extra por ciclo. Para tareas
  simples de 1–2 herramientas, `planning_interval=None` es más eficiente.


---
## Conclusiones

### Ventajas de Agentic RAG frente a RAG tradicional

| RAG Tradicional | Agentic RAG (este proyecto) |
|---|---|
| Recupera documentos y genera respuesta en un solo paso | El agente decide dinámicamente cuándo y qué recuperar |
| No puede complementar con búsqueda web | Combina base interna + web según la consulta |
| No puede realizar cálculos | Usa tools especializadas (Haversine) |
| Sin capacidad de razonamiento multi-paso | Ciclo Thought → Code → Observation iterativo |
| Una sola fuente de conocimiento | Múltiples fuentes integradas automáticamente |

### Lo que aprendimos
1. La **descripción del tool** es el contrato entre el LLM y la herramienta — escribirla bien es tan importante como la implementación.
2. **BM25** es un retriever rápido y sin dependencias de GPU, ideal para bases de conocimiento pequeñas y medianas.
3. El **planning periódico** mejora la coherencia en tareas multi-objetivo a costa de más tokens.
4. La **subclase de Tool** permite encapsular estado (retriever, modelos, conexiones) de forma limpia y reutilizable.
